In [ ]:
# SETUP
from google.colab import drive
drive.mount('/content/drive')

!pip install mne --quiet
import os
import mne
import matplotlib.pyplot as plt
import gc
import traceback

In [ ]:
import os
import mne

# Root directory
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'

# Loop through participant folders
subject_dirs = sorted([d for d in os.listdir(root_dir) if d.isdigit()], key=lambda x: int(x))

for subj in subject_dirs:
    print(f"\n🧠 Processing participant {subj}")
    subj_path = os.path.join(root_dir, subj)
    epo_path = os.path.join(subj_path, f"{subj}_epo.fif")
    clean_epo_path = os.path.join(subj_path, f"{subj}_epo_clean.fif")

    if not os.path.exists(epo_path):
        print(f"❌ No epoch file found for participant {subj}. Skipping.")
        continue

    if os.path.exists(clean_epo_path):
        print(f"✅ Cleaned epochs already exist for {subj}. Skipping.")
        continue

    try:
        epochs = mne.read_epochs(epo_path, preload=True)

        # Apply rejection: e.g., reject if any EEG channel exceeds ±150 µV
        epochs.drop_bad(reject=dict(eeg=150e-6))

        # Save cleaned epochs
        epochs.save(clean_epo_path, overwrite=True)
        print(f"💾 Cleaned epochs saved: {clean_epo_path}")

    except Exception as e:
        print(f"⚠️ Error processing participant {subj}: {e}")


In [ ]:
import os
import mne
import matplotlib.pyplot as plt

# === Configuration ===
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'
participant = "3"  # ← Change this or loop over more participants
epo_path = os.path.join(root_dir, participant, f"{participant}_epo.fif")

# List of thresholds to test (in volts)
thresholds = [100e-6, 125e-6, 150e-6, 175e-6, 200e-6]
drop_rates = []

# === Load epochs ===
if not os.path.exists(epo_path):
    print(f"❌ Epoch file not found for participant {participant}")
else:
    print(f"📂 Loaded: {epo_path}")
    for th in thresholds:
        try:
            # Reload epochs fresh each time
            epochs = mne.read_epochs(epo_path, preload=True)
            n_total = len(epochs)

            # Apply threshold rejection
            epochs.drop_bad(reject=dict(eeg=th))
            n_clean = len(epochs)
            drop_rate = 1 - (n_clean / n_total)
            drop_rates.append(drop_rate)

            print(f"🔍 Threshold: {int(th*1e6)} µV — Dropped {drop_rate*100:.1f}% of epochs")

        except Exception as e:
            print(f"⚠️ Error at threshold {int(th*1e6)} µV: {e}")
            drop_rates.append(None)

    # === Plot results ===
    plt.figure(figsize=(8, 5))
    plt.plot([int(t*1e6) for t in thresholds], [r * 100 for r in drop_rates], marker='o')
    plt.xlabel('Threshold (µV)')
    plt.ylabel('Dropped Epochs (%)')
    plt.title(f'Epoch Rejection Rate — Participant {participant}')
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
# + plot rejected ones